# PRONOSTIA Clean Rebuild Pipeline

### Import Colab setup utilities

In [1]:
from pathlib import Path
import os
import sys
import json
import hashlib
import shutil
import subprocess
import importlib.util
import zipfile
import urllib.request
import uuid
from datetime import datetime, timezone


### Detect the Colab environment

In [2]:
try:
    IS_COLAB = importlib.util.find_spec('google.colab') is not None
except ModuleNotFoundError:
    IS_COLAB = False
print('Runtime:', 'Google Colab' if IS_COLAB else 'Local Jupyter')


Runtime: Local Jupyter


### Configure Colab paths

In [3]:
COLAB_MOUNT_DRIVE = False # @param {type:"boolean"}
COLAB_BASE_DIR = Path(os.environ.get('COLAB_BASE_DIR', Path.cwd()))
COLAB_WORK_DIR = str(COLAB_BASE_DIR / 'bearing_rul' / 'handover_notebook_only')
COLAB_BUNDLE_ZIP = str(COLAB_BASE_DIR / 'handover_notebook_only.zip')
COLAB_RAW_SOURCE = 'download' # @param ["download", "drive_zip", "existing"]
COLAB_RAW_ROOT = str(COLAB_BASE_DIR / 'datasets' / 'pronostia' / 'ieee-phm-2012-data-challenge-dataset-master')
COLAB_OUTPUT_BASE = str(COLAB_BASE_DIR / 'bearing_rul_outputs')
COLAB_DRIVE_ROOT = COLAB_BASE_DIR / 'drive'
COLAB_DRIVE_ARCHIVE = str(COLAB_DRIVE_ROOT / 'MyDrive' / 'Bearing_RUL' / 'phm2012-master.zip')
COLAB_DOWNLOAD_ARCHIVE = str(COLAB_BASE_DIR / 'datasets' / 'pronostia' / 'phm2012-master.zip')


### Mount Google Drive when enabled

In [4]:
if IS_COLAB and (COLAB_MOUNT_DRIVE or COLAB_RAW_SOURCE=='drive_zip'):
    from google.colab import drive
    drive.mount(str(COLAB_DRIVE_ROOT))
else:
    print('Không mount Drive; dùng filesystem hiện tại.')


Không mount Drive; dùng filesystem hiện tại.


### Prepare the Colab notebook directory

In [5]:
if IS_COLAB:
    Path(COLAB_WORK_DIR).mkdir(parents=True,exist_ok=True)
    os.chdir(COLAB_WORK_DIR)
print('Working directory:',Path.cwd())


Working directory: <PROJECT_ROOT>


### Receive the notebook bundle on Colab

In [6]:
required_bundle_files = ['data_process.ipynb','data_pipeline_scripts.ipynb','pronostia_config.yaml','test_ground_truth.csv','requirements.txt','README.md']
if IS_COLAB and not all(Path(name).is_file() for name in required_bundle_files):
    bundle_path = Path(COLAB_BUNDLE_ZIP)
    if not bundle_path.is_file():
        from google.colab import files
        print('Chọn handover_notebook_only.zip, KHÔNG chọn raw ZIP ở bước này.')
        uploaded = files.upload()
        candidates = [name for name in uploaded if name.lower().endswith('.zip')]
        assert len(candidates)==1, 'Cần đúng một ZIP bộ notebook'
        bundle_path = Path(candidates[0])
        bundle_path.write_bytes(uploaded[candidates[0]])
else:
    bundle_path = None


### Extract the required notebook files

In [7]:
if IS_COLAB and bundle_path is not None:
    with zipfile.ZipFile(bundle_path) as bundle:
        for filename in required_bundle_files:
            candidates=[name for name in bundle.namelist() if name in (filename,'handover_notebook_only/'+filename)]
            assert len(candidates)==1, f'Thiếu/trùng file trong ZIP: {filename}'
            Path(filename).write_bytes(bundle.read(candidates[0]))
    print('Đã nhận đầy đủ notebook/config/dependencies. Bỏ qua reports cũ trong ZIP.')


### Install dependencies on Colab

In [8]:
if IS_COLAB:
    subprocess.check_call([sys.executable,'-m','pip','install','--quiet','-r','requirements.txt'])
else:
    print('Local: dùng môi trường đã cài; pip install -r requirements.txt khi cần.')


Local: dùng môi trường đã cài; pip install -r requirements.txt khi cần.


### Import notebook utilities

In [9]:
from pathlib import Path
import os
import sys
import json
import platform
import uuid
from datetime import datetime
import nbformat


### Load definitions from the data notebook

In [10]:
def find_bundle_root(start=None):
    override = os.environ.get('PRONOSTIA_PROJECT_ROOT')
    start = Path(start or Path.cwd()).resolve()
    candidates = ([Path(override).expanduser().resolve()] if override else []) + [start, *start.parents]
    for base in candidates:
        for candidate in (base, base / 'handover_notebook_only', base / 'customer_handover'):
            if (candidate / 'pronostia_config.yaml').is_file() and (candidate / 'data_process.ipynb').is_file():
                return candidate.resolve()
    raise FileNotFoundError('Không tìm thấy bộ notebook. Hãy mở Jupyter từ thư mục bàn giao hoặc đặt PRONOSTIA_PROJECT_ROOT.')

PROJECT_ROOT = find_bundle_root()
data_path = PROJECT_ROOT / 'data_process.ipynb'
assert data_path.is_file(), f'Thiếu notebook cùng thư mục: {data_path}'
data_notebook = nbformat.read(data_path,as_version=4)
definition_phases = {'imports','config','definition','test_definition'}
for cell in data_notebook.cells:
    if cell.cell_type=='code' and cell.metadata.get('phase') in definition_phases:
        exec(compile(cell.source,'data_process.ipynb: '+cell.metadata.get('task','definition'),'exec'),globals())


### Configure paths and execution mode

In [11]:
RAW_ROOT = Path(os.environ.get('PRONOSTIA_RAW_ROOT') or CONFIG.get('raw_root') or
                (Path(COLAB_RAW_ROOT) if IS_COLAB else PROJECT_ROOT / 'data' / 'raw' / 'pronostia')).expanduser().resolve()
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')+'_'+uuid.uuid4().hex[:8]
OUTPUT_ROOT = Path(os.environ.get('PRONOSTIA_OUTPUT_ROOT') or CONFIG.get('output_root') or (Path(COLAB_OUTPUT_BASE)/RUN_ID if IS_COLAB else PROJECT_ROOT/'runs'/RUN_ID)).resolve()
EXECUTION_MODE = os.environ.get('PRONOSTIA_EXECUTION_MODE',CONFIG['execution_mode'])
REPORTS_ROOT = OUTPUT_ROOT/'reports' if IS_COLAB else PROJECT_ROOT/'reports'
RUN_CONFIG = copy.deepcopy(CONFIG)
RUN_CONFIG.update(raw_root=str(RAW_ROOT),output_root=str(OUTPUT_ROOT))
assert EXECUTION_MODE in ('clean_rebuild','validate_existing')
assert RUN_CONFIG['max_files'] is None


### Download the pinned raw dataset ZIP

In [12]:
if IS_COLAB and COLAB_RAW_SOURCE=='download' and not RAW_ROOT.exists():
    raw_archive = Path(COLAB_DOWNLOAD_ARCHIVE)
    if not raw_archive.exists():
        raw_archive.parent.mkdir(parents=True,exist_ok=True)
        partial=raw_archive.with_suffix('.zip.part')
        with urllib.request.urlopen(CONFIG['dataset_source']['archive_url'],timeout=120) as response, partial.open('wb') as target:
            shutil.copyfileobj(response,target,length=4*1024*1024)
        partial.replace(raw_archive)
    print('Archive:',raw_archive)
elif IS_COLAB and COLAB_RAW_SOURCE=='drive_zip' and not RAW_ROOT.exists():
    raw_archive=Path(COLAB_DRIVE_ARCHIVE)
    assert raw_archive.is_file(), f'Upload raw ZIP vào Drive trước: {raw_archive}'
else:
    raw_archive=None


### Verify the raw ZIP checksum and commit

In [13]:
archive_verification=None
if raw_archive is not None:
    archive_verification={'sha256':verify_pinned_archive(raw_archive,CONFIG['dataset_source']),
                          'commit_sha':CONFIG['dataset_source']['commit_sha'],'status':'PASS'}
    print(archive_verification)


### Extract raw data into a new directory

In [14]:
if raw_archive is not None:
    assert archive_verification['status']=='PASS'
    assert not RAW_ROOT.exists(), 'Không ghi đè raw đang có'
    stage=RAW_ROOT.parent/(RAW_ROOT.name+'.extracting_'+uuid.uuid4().hex[:8])
    stage.mkdir(parents=True)
    with zipfile.ZipFile(raw_archive) as archive:
        top_levels={name.split('/')[0] for name in archive.namelist() if name}
        assert len(top_levels)==1, 'Archive phải có đúng một thư mục gốc'
        for member in archive.infolist():
            parts=Path(member.filename).parts[1:]
            if not parts:continue
            destination=(stage.joinpath(*parts)).resolve()
            assert destination.is_relative_to(stage.resolve()), 'Unsafe archive path'
            if member.is_dir():destination.mkdir(parents=True,exist_ok=True)
            else:
                destination.parent.mkdir(parents=True,exist_ok=True)
                with archive.open(member) as source,destination.open('wb') as target:shutil.copyfileobj(source,target)
    stage.rename(RAW_ROOT)
    print('Raw extracted:',RAW_ROOT)


### Validate the pipeline raw data path

In [15]:
assert RAW_ROOT.is_dir(), f'Đặt RAW_ROOT/PRONOSTIA_RAW_ROOT: {RAW_ROOT}'
assert all((RAW_ROOT/subset).is_dir() for subset in ('Learning_set','Test_set','Full_Test_Set')), 'Raw cần đủ ba tập'
print('Raw dataset path: PASS')


Raw dataset path: PASS


### Display the runtime environment

In [16]:
ENVIRONMENT = {'python':platform.python_version(),'numpy':np.__version__,'pandas':pd.__version__,
               'torch':torch.__version__,'scipy':__import__('scipy').__version__,'yaml':yaml.__version__,'platform':platform.platform(),'runtime':'google_colab' if IS_COLAB else 'local',
               'started_at_utc':datetime.now(timezone.utc).isoformat(),
               'colab_release_tag':os.environ.get('COLAB_RELEASE_TAG'),
               'colab_module_available':IS_COLAB}
print('PROJECT_ROOT:',PROJECT_ROOT)
print('RAW_ROOT:',RAW_ROOT)
print('OUTPUT_ROOT:',OUTPUT_ROOT)
display(pd.DataFrame([ENVIRONMENT]))


PROJECT_ROOT: <PROJECT_ROOT>
RAW_ROOT: <PROJECT_ROOT>\data\raw\pronostia
OUTPUT_ROOT: <PROJECT_ROOT>\artifacts\pronostia_cache_rebuilt
   python  numpy pandas       torch   scipy   yaml                   platform runtime                    started_at_utc colab_release_tag  colab_module_available
0  3.13.5  2.2.4  2.2.3  2.12.1+cpu  1.15.2  6.0.3  Windows-11-10.0.26200-SP0   local  2026-10-23T12:01:56.204072+00:00              None                   False


### Set the random seed

In [17]:
np.random.seed(CONFIG['seed'])
torch.manual_seed(CONFIG['seed'])


### Limit CPU threads

In [18]:
torch.set_num_threads(2)


### Validate the output directory

In [19]:
if EXECUTION_MODE=='clean_rebuild':
    if OUTPUT_ROOT.exists() and any(OUTPUT_ROOT.iterdir()):
        raise FileExistsError('clean_rebuild yêu cầu output mới/rỗng; không tự xóa cache cũ')
    OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
else:
    assert (OUTPUT_ROOT/'manifest_all_files.csv').is_file(), 'validate_existing cần output cache hợp lệ'
REPORTS_ROOT.mkdir(exist_ok=True)


### Prepare the test fixture directory

In [20]:
Path('.tmp').mkdir(exist_ok=True)


### Run parser tests

In [21]:
parser_test_result = execute_test_classes([ParserTests])


.......
----------------------------------------------------------------------
Ran 7 tests in 0.091s

OK


### Run contract tests

In [22]:
contract_test_result = execute_test_classes([ContractTests])


.......
----------------------------------------------------------------------
Ran 7 tests in 0.049s

OK


### Run identity tests

In [23]:
identity_test_result = execute_test_classes([CacheIdentityTests])


.........
----------------------------------------------------------------------
Ran 9 tests in 0.196s

OK


### Run regression tests

In [24]:
feedback_test_result = execute_test_classes([FeedbackRegressionTests])


.............
----------------------------------------------------------------------
Ran 13 tests in 0.031s

OK


### Run integration tests

In [25]:
integration_test_result = execute_test_classes([FixtureIntegrationTests])


......
----------------------------------------------------------------------
Ran 6 tests in 4.814s

OK


### Aggregate test results

In [26]:
TEST_RESULTS = {'parser':parser_test_result, 'contract':contract_test_result, 'identity':identity_test_result, 'feedback':feedback_test_result, 'integration':integration_test_result}
display(pd.DataFrame.from_dict(TEST_RESULTS,orient='index'))


             tests  failures  errors status
parser           7         0       0   PASS
contract         7         0       0   PASS
identity         9         0       0   PASS
feedback        13         0       0   PASS
integration      6         0       0   PASS


### Load sourced ground truth

In [27]:
GROUND_TRUTH = pd.read_csv(PROJECT_ROOT/'test_ground_truth.csv')
assert set(GROUND_TRUTH.bearing_id)==set(TEST)
assert GROUND_TRUTH.source.notna().all() and GROUND_TRUTH.source_locator.notna().all()
assert dict(zip(GROUND_TRUTH.bearing_id,GROUND_TRUTH.rul_seconds))==RUN_CONFIG['test_rul_at_cut_seconds']


### Scan the dataset structure

In [28]:
tasks, structure_issues = scan_dataset_structure(RAW_ROOT)
assert not any(i['severity']=='ERROR' for i in structure_issues),structure_issues


### Scan file schemas and checksums

In [29]:
records = scan_inventory_files(RAW_ROOT,tasks,RUN_CONFIG)
print('Raw CSV:',len(records))


Raw CSV: 43592


### Validate filename indices

In [30]:
index_issues = validate_filename_indices(records)


### Validate learning terminal completeness

In [31]:
validate_learning_terminals(records,RUN_CONFIG['expected_learning_terminal_indices'])


### Build the acquisition manifest

In [32]:
manifest, clock_repairs = build_acquisition_records(records,RUN_CONFIG)


### Analyze data quality

In [33]:
quality = analyze_inventory_quality(records,structure_issues,index_issues,clock_repairs)
assert not any(i['severity']=='ERROR' for i in quality['issues']),quality['issues']


### Export inventory manifests

In [34]:
if EXECUTION_MODE=='clean_rebuild':
    pd.DataFrame(records).to_csv(OUTPUT_ROOT/'manifest_all_files.csv',index=False)
    manifest.to_csv(OUTPUT_ROOT/'manifest.csv',index=False)


### Export the source recipe

In [35]:
if EXECUTION_MODE=='clean_rebuild':
    (OUTPUT_ROOT/'preprocessing.yaml').write_text(yaml.safe_dump(RUN_CONFIG,sort_keys=True),encoding='utf-8')


### Compare ground truth with Full_Test_Set

In [36]:
rul_audit = ground_truth_comparison(manifest,GROUND_TRUTH)
display(rul_audit)


    bearing_id  rul_seconds                                   source                                                                                  source_locator  verified_against_full_test                                                                                                                                                primary_source                                              primary_source_locator  verified_against_primary_source  test_cut_timestamp  full_test_terminal_timestamp  derived_rul_seconds  configured_rul_seconds  absolute_difference_seconds    verification_status
0   Bearing1_3         5730  https://www.mdpi.com/2076-3417/8/7/1102  Table 2, DOI 10.3390/app8071102; secondary published benchmark table, not organizer answer-key                        True  https://raw.githubusercontent.com/Lucky-Loek/ieee-phm-2012-data-challenge-dataset/129d43e0aa71e1da992bb4c0f69d54fd990c58cf/IEEEPHM2012-Challenge-Details.pdf  Organizer challenge document; Section 5.2, Table 

### Export the ground truth audit

In [37]:
rul_audit.to_csv(REPORTS_ROOT/'ground_truth_full_test_comparison.csv',index=False)


### Select model input bearings

In [38]:
frames = bearing_frames(manifest)
assert not any(f.subset.iloc[0]=='Full_Test_Set' for f in frames.values())


### Create absolute RUL labels

In [39]:
label_frames = {b:label_bearing(frame,RUN_CONFIG) for b,frame in frames.items()}


### Load temperature streams

In [40]:
streams = {b:load_bearing_temperature(RAW_ROOT,records,frame,RUN_CONFIG) for b,frame in label_frames.items()}


### Align causal temperature

In [41]:
aligned = {b:align_bearing_temperature(streams[b],frame,RUN_CONFIG) for b,frame in label_frames.items()}


### Validate temperature alignment

In [42]:
for values in aligned.values(): validate_temperature_alignment(values,RUN_CONFIG)


### Add label provenance to the manifest

In [43]:
for b,frame in label_frames.items():
    rows=manifest.index[(manifest.bearing_id==b)&(manifest.subset==frame.subset.iloc[0])]
    assert len(rows)==len(frame)
    manifest.loc[rows,'rul_seconds']=frame.rul_seconds.to_numpy()
    manifest.loc[rows,'rul_cycles']=frame.rul_cycles.to_numpy()
    manifest.loc[rows,'temp_timestamp']=aligned[b]['matched']
    manifest.loc[rows,'alignment_delta_s']=aligned[b]['delta']
    manifest.loc[rows,'temp_missing_mask']=aligned[b]['temperature_mask'].reshape(len(frame),-1).all(axis=1)


### Export the enriched acquisition manifest

In [44]:
if EXECUTION_MODE=='clean_rebuild':
    manifest.to_csv(OUTPUT_ROOT/'manifest.csv',index=False)


### Transform bearing vibration data

In [45]:
vibration = {b:transform_bearing_vibration(RAW_ROOT,frame,RUN_CONFIG) for b,frame in label_frames.items()}


### Assemble acquisition shards

In [46]:
arrays = {b:assemble_bearing_shard(frame,vibration[b],aligned[b]) for b,frame in label_frames.items()}


### Extract handcrafted baseline features


In [47]:
arrays = {bearing:add_handcrafted_features(values,RUN_CONFIG) for bearing,values in arrays.items()}


### Display handcrafted baseline features


In [48]:
feature_preview = pd.DataFrame(arrays[next(iter(arrays))]['handcrafted_features'][:5], columns=handcrafted_feature_names(RUN_CONFIG))
display(feature_preview)


   horizontal_rms  horizontal_kurtosis  horizontal_crest_factor  horizontal_skewness  horizontal_peak_to_peak  horizontal_spectral_kurtosis_mean  horizontal_spectral_kurtosis_max  horizontal_spectral_kurtosis_peak_hz  horizontal_bpfo_h1_relative_bandpower  horizontal_bpfo_h2_relative_bandpower  horizontal_bpfo_h3_relative_bandpower  horizontal_bpfi_h1_relative_bandpower  horizontal_bpfi_h2_relative_bandpower  horizontal_bpfi_h3_relative_bandpower  horizontal_bsf_h1_relative_bandpower  horizontal_bsf_h2_relative_bandpower  horizontal_bsf_h3_relative_bandpower  vertical_rms  vertical_kurtosis  vertical_crest_factor  vertical_skewness  vertical_peak_to_peak  vertical_spectral_kurtosis_mean  vertical_spectral_kurtosis_max  vertical_spectral_kurtosis_peak_hz  vertical_bpfo_h1_relative_bandpower  vertical_bpfo_h2_relative_bandpower  vertical_bpfo_h3_relative_bandpower  vertical_bpfi_h1_relative_bandpower  vertical_bpfi_h2_relative_bandpower  vertical_bpfi_h3_relative_bandpower  vertical_bsf_

### Export the handcrafted feature schema


In [49]:
write_json(REPORTS_ROOT/'handcrafted_feature_schema.json',handcrafted_feature_schema(RUN_CONFIG))


### Export acquisition shards

In [50]:
shards = export_acquisition_shards(OUTPUT_ROOT,arrays) if EXECUTION_MODE=='clean_rebuild' else {b:OUTPUT_ROOT/'acquisitions'/f'{b}.npz' for b in arrays}


### Calculate temperature coverage

In [51]:
coverage = temperature_coverage(records,label_frames,streams,aligned)
display(coverage)


    bearing_id        subset condition_id  acc_file_count  temp_file_count  first_acc_timestamp  last_acc_timestamp  first_temp_timestamp  last_temp_timestamp  aligned_acquisition_count  aligned_acquisition_percent  missing_element_percent  high_warning
0   Bearing1_1  Learning_set          OC1            2803              466         34779.065664        62799.065664               34847.5              62799.1                       2796                    99.750268                 0.374599         False
1   Bearing1_2  Learning_set          OC1             871              144         31625.196910        40325.196910               31685.7              40325.2                        864                    99.196326                 1.115815         False
2   Bearing2_1  Learning_set          OC2             911              151         29655.884410        38755.884410               29752.3              38755.9                        901                    98.902305                 1.45787

### Export temperature coverage

In [52]:
coverage.to_csv(REPORTS_ROOT/'temperature_coverage.csv',index=False)


### Analyze missing modalities by fold

In [53]:
split_protocols = protocols(RUN_CONFIG['seed'])
splits = split_protocols['development_cv']+[split_protocols['official_benchmark']]
missing_bearings = coverage.loc[coverage.high_warning,'bearing_id'].tolist()
fold_missing = [{'fold':str(s['fold']),'train_missing_temperature':[b for b in s['train'] if b in missing_bearings],
                'validation_missing_temperature':[b for b in s['validation'] if b in missing_bearings],
                'test_missing_temperature':[b for b in s['test'] if b in missing_bearings]} for s in splits]
display(pd.DataFrame(fold_missing))


       fold train_missing_temperature validation_missing_temperature              test_missing_temperature
0         0  [Bearing2_2, Bearing3_2]                             []                                    []
1         1  [Bearing2_2, Bearing3_2]                             []                                    []
2         2  [Bearing2_2, Bearing3_2]                             []                                    []
3         3              [Bearing3_2]                   [Bearing2_2]                                    []
4         4  [Bearing2_2, Bearing3_2]                             []                                    []
5         5              [Bearing2_2]                   [Bearing3_2]                                    []
6  official  [Bearing2_2, Bearing3_2]                             []  [Bearing1_3, Bearing2_3, Bearing2_6]


### Export the missing modality protocol

In [54]:
write_json(REPORTS_ROOT/'modality_protocol.json',{
 'folds':fold_missing,
 'fully_missing_bearings':missing_bearings,
 'official_test_fully_missing':int(((coverage.subset=='Test_set')&coverage.high_warning).sum()),
 'official_test_partially_missing':int(((coverage.subset=='Test_set')&(coverage.missing_element_percent>0)&~coverage.high_warning).sum()),
 'missing_temperature_policy':RUN_CONFIG['temperature']['missing_policy']})


### Create the temperature report

In [55]:
temperature_report = make_temperature_report(coverage,aligned)


### Create the label report

In [56]:
label_report = make_label_report(label_frames)


### Export label and temperature reports

In [57]:
write_json(REPORTS_ROOT/'label_report.json',label_report)
write_json(REPORTS_ROOT/'temperature_report.json',temperature_report)


### Export split protocols

In [58]:
if EXECUTION_MODE=='clean_rebuild':
    (OUTPUT_ROOT/'splits.yaml').write_text(yaml.safe_dump(split_protocols,sort_keys=True),encoding='utf-8')


### Resolve fold directories

In [59]:
folders = {str(s['fold']):OUTPUT_ROOT/('official' if s['fold']=='official' else f"fold_{s['fold']}") for s in splits}


### Load or fit training-only fold scalers

In [60]:
scalers = {str(s['fold']):fit_scaler(shards,s['train'],fingerprint(RUN_CONFIG),fingerprint(s),RUN_CONFIG['scaler_epsilon']) if EXECUTION_MODE=='clean_rebuild' else json.loads((folders[str(s['fold'])]/'scaler.json').read_text()) for s in splits}


### Export fold scaler provenance

In [61]:
if EXECUTION_MODE=='clean_rebuild':
    for split in splits:
        folder=folders[str(split['fold'])];folder.mkdir(exist_ok=True)
        write_json(folder/'scaler.json',scalers[str(split['fold'])])
        write_json(folder/'split.json',split)


### Export normalized fold shards

In [62]:
if EXECUTION_MODE=='clean_rebuild':
    for split in splits:
        export_scaled_shards(folders[str(split['fold'])],scale_fold_shards(shards,split,scalers[str(split['fold'])]))


### Export window metadata

In [63]:
if EXECUTION_MODE=='clean_rebuild':
    for split in splits:export_window_metadata(folders[str(split['fold'])],split,RUN_CONFIG)


### Load or build fold cache indices

In [64]:
cache_indices = {str(s['fold']):build_cache_index(folders[str(s['fold'])],s) if EXECUTION_MODE=='clean_rebuild' else pd.read_csv(folders[str(s['fold'])]/'cache_index.csv') for s in splits}


### Export fold cache indices

In [65]:
if EXECUTION_MODE=='clean_rebuild':
    for key,index in cache_indices.items():
        index.to_csv(folders[key]/'cache_index.csv',index=False)
        index.to_parquet(folders[key]/'cache_index.parquet',index=False)


### Export fold manifests

In [ ]:
if EXECUTION_MODE=='clean_rebuild':
    for split in splits:fold_manifest(manifest,split).to_csv(folders[str(split['fold'])]/'manifest.csv',index=False)


### Export cache identities

In [ ]:
if EXECUTION_MODE=='clean_rebuild':
    for split in splits:write_json(folders[str(split['fold'])]/'cache_identity.json',cache_identity(quality['source_hash'],RUN_CONFIG,split,scalers[str(split['fold'])]))


### Validate cache identities

In [ ]:
for folder in folders.values():verify_identity(folder,expected_config=RUN_CONFIG,verified_source_hash=quality['source_hash'])


### Validate split and scaler invariants

In [ ]:
for folder in folders.values():validate_split_scaler(folder)


### Validate shard labels and clocks

In [ ]:
validated_shards={}
for split in splits:
    key=str(split['fold'])
    for role in ('train','validation','test'):
        for b in split[role]:
            validate_one_shard(folders[key]/'shards'/role/f'{b}.npz',scalers[key]['hash'],role!='test')


### Validate window boundaries

In [ ]:
for split in splits:
    for role in ('train','validation','test'):
        for b in split[role]:
            with np.load(folders[str(split['fold'])]/'shards'/role/f'{b}.npz',allow_pickle=False) as z:
                validate_window_shard(z,role,RUN_CONFIG)


### Validate scaled temperature masks

In [ ]:
for split in splits:
    for role in ('train','validation','test'):
        for b in split[role]:
            with np.load(folders[str(split['fold'])]/'shards'/role/f'{b}.npz',allow_pickle=False) as z:validate_temperature_shard(z)


### Validate full-pass DataLoaders

In [ ]:
loader_results={str(s['fold']):{role:validate_one_loader(folders[str(s['fold'])],role,RUN_CONFIG)
                 for role in ('train','validation','test') if s[role]} for s in splits}


### Benchmark DataLoader batches

In [ ]:
benchmarks={str(s['fold']):{role:benchmark_loader(folders[str(s['fold'])],role)
             for role in ('train','validation','test') if s[role]} for s in splits}


### Aggregate subset-condition-split counts

In [ ]:
fold_counts = {str(s['fold']):aggregate_fold_counts(records,manifest,cache_indices[str(s['fold'])],s) for s in splits}


### Measure process memory

In [ ]:
ram = {'rss_mb':psutil.Process().memory_info().rss/2**20}


### Assemble the validation report

In [ ]:
validation_report={'status':'PASS','errors':[], 'folds':{key:{'status':'PASS','full_pass':loader_results[key],
 'benchmarks':benchmarks[key],'counts':fold_counts[key]} for key in folders},'environment':ENVIRONMENT,'ram':ram,
 'high_warnings':{'temperature_missing':missing_bearings,'ground_truth_full_test_mismatches':rul_audit.loc[~rul_audit.verified_against_full_test,'bearing_id'].tolist()},
 'official_benchmark_ground_truth_ready':bool(rul_audit.verified_against_full_test.all())}


### Export validation evidence

In [ ]:
write_json(REPORTS_ROOT/'validation_report.json',validation_report)
for key,value in validation_report['folds'].items():write_json(folders[key]/'validation_report.json',value)


### Export the quality report

In [ ]:
write_json(REPORTS_ROOT/'data_quality_report.json',quality)


### Select learning-only QC samples

In [ ]:
qc_samples = select_qc_samples(RAW_ROOT,manifest,RUN_CONFIG)


### Visualize waveforms

In [ ]:
waveform_figure = plot_waveform_qc(qc_samples)
display(waveform_figure)


### Export the waveform figure

In [ ]:
waveform_figure.savefig(REPORTS_ROOT/'waveform.png',dpi=150)
plt.close(waveform_figure)


### Visualize the power spectral density

In [ ]:
psd_figure = plot_psd_qc(qc_samples)
display(psd_figure)


### Export the power spectral density figure

In [ ]:
psd_figure.savefig(REPORTS_ROOT/'psd.png',dpi=150)
plt.close(psd_figure)


### Visualize RMS values

In [ ]:
rms_figure = plot_rms_qc(shards)
display(rms_figure)


### Export the RMS figure

In [ ]:
rms_figure.savefig(REPORTS_ROOT/'rms.png',dpi=150)
plt.close(rms_figure)


### Visualize the temperature histogram

In [ ]:
temperature_histogram_figure = plot_temperature_histogram(streams)
display(temperature_histogram_figure)


### Export the temperature histogram

In [ ]:
temperature_histogram_figure.savefig(REPORTS_ROOT/'temperature_histogram.png',dpi=150)
plt.close(temperature_histogram_figure)


### Visualize the alignment histogram

In [ ]:
alignment_histogram_figure = plot_alignment_histogram(aligned)
display(alignment_histogram_figure)


### Export the alignment histogram

In [ ]:
alignment_histogram_figure.savefig(REPORTS_ROOT/'alignment_histogram.png',dpi=150)
plt.close(alignment_histogram_figure)


### Calculate anti-aliasing quality checks

In [ ]:
aliasing_qc = calculate_aliasing_qc(RUN_CONFIG['vibration'])


### Validate the anti-aliasing invariant

In [ ]:
assert aliasing_qc['pass'],aliasing_qc


### Export preprocessing quality checks

In [ ]:
write_json(REPORTS_ROOT/'preprocessing_qc.json',{'aliasing_check':aliasing_qc,'figures':['waveform.png','psd.png','rms.png','temperature_histogram.png','alignment_histogram.png']})


### Display the fold summary

In [ ]:
fold_summary=[{'fold':key,'status':'PASS',**{role:loader_results[key].get(role,{}).get('windows',0) for role in ('train','validation','test')}} for key in folders]
display(pd.DataFrame(fold_summary))


### Display official split counts

In [ ]:
display(pd.DataFrame(fold_counts['official']))


### Assemble the clean rebuild summary

In [ ]:
execution_summary={'preprocessing_status':'PASS','rebuilt':EXECUTION_MODE=='clean_rebuild','execution_mode':EXECUTION_MODE,
 'raw_root':str(RAW_ROOT),'output_root':str(OUTPUT_ROOT),'environment':ENVIRONMENT,'tests':TEST_RESULTS,'folds':fold_summary,
 'source_hash':quality['source_hash'],'config_hash':fingerprint(RUN_CONFIG),'dataset_source':RUN_CONFIG['dataset_source'],
 'ground_truth_comparison':rul_audit.to_dict('records'),'official_benchmark_ground_truth_ready':bool(rul_audit.verified_against_full_test.all()),
 'acceptance_status':'REVIEW_REQUIRED_GROUND_TRUTH' if not rul_audit.verified_against_full_test.all() else 'PASS',
 'modality_missing_bearings':missing_bearings,'aliasing_check':aliasing_qc}


### Export clean rebuild evidence

In [ ]:
write_json(REPORTS_ROOT/'clean_rebuild_execution.json',execution_summary)
write_json(REPORTS_ROOT/'notebook_test_report.json',TEST_RESULTS)


### Export the compact manifest summary

In [ ]:
pd.DataFrame(records).groupby(['subset','bearing_id','condition_id','modality'],sort=True).size().rename('file_count').reset_index().to_csv(REPORTS_ROOT/'manifest_summary.csv',index=False)


### Display the final acceptance status

In [ ]:
print('preprocessing_status:',execution_summary['preprocessing_status'])
print('rebuilt:',execution_summary['rebuilt'])
print('tests:',sum(row['tests'] for row in TEST_RESULTS.values()))
print('acceptance_status:',execution_summary['acceptance_status'])
print('official_benchmark_ground_truth_ready:',execution_summary['official_benchmark_ground_truth_ready'])
print('Reports:',REPORTS_ROOT)


### Create the Google Colab execution report

In [ ]:
if IS_COLAB:
    colab_report={**execution_summary,'report_kind':'actual_google_colab_execution','runtime':'google_colab',
        'finished_at_utc':datetime.now(timezone.utc).isoformat(),'archive_verification':archive_verification,
        'notebook':'data_pipeline_scripts.ipynb','test_count':sum(r['tests'] for r in TEST_RESULTS.values()),
        'validation_status':validation_report['status'],'full_pass':loader_results,
        'notebook_source_sha256':{name:hashlib.sha256((PROJECT_ROOT/name).read_bytes()).hexdigest()
                                  for name in ('data_process.ipynb','data_pipeline_scripts.ipynb')}}
    write_json(REPORTS_ROOT/'colab_execution.json',colab_report)
    print('Actual Colab execution report:',REPORTS_ROOT/'colab_execution.json')
else:
    print('Local execution; không ghi colab_execution.json.')


### Package reports from the Colab run

In [ ]:
if IS_COLAB:
    data_proof=PROJECT_ROOT/'reports/data_notebook_colab_execution.json'
    if data_proof.exists():shutil.copy2(data_proof,REPORTS_ROOT/data_proof.name)
    report_zip=Path(shutil.make_archive(str(OUTPUT_ROOT/'colab_reports'),'zip',root_dir=REPORTS_ROOT))
    print('Reports ZIP:',report_zip)


### Download the Colab execution report

In [ ]:
if IS_COLAB:
    from google.colab import files
    files.download(str(report_zip))
